In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

xlsx_path = r"F:\博士\ITS Ghana\ITS_Survey_Cleaned_Numeric.xlsx"
df = pd.read_excel(xlsx_path)

HAS_ITS_TECH_FAM = "ITS_Tech_Fam" in df.columns

# Measurement blocks (reflective)
blocks = {
    "FAM": ["Familiarity"] + (["ITS_Tech_Fam"] if HAS_ITS_TECH_FAM else []),
    "ATT": ["Improve_ITS","Learn_More"],
    "BI" : ["Would_Use_ITS"],
    "USE": ["Apps_Use_Freq"],
}

# Build latent scores as simple averages (external estimation)
for lv, inds in blocks.items():
    df[lv] = df[inds].mean(axis=1)

dat = df[list(blocks.keys())].dropna().copy()

# Structural regressions approximating PLS inner model
def ols(y, X):
    Xc = sm.add_constant(X)
    mod = sm.OLS(y, Xc, missing='drop').fit()
    return mod

# USE -> FAM
m1 = ols(dat["FAM"], dat[["USE"]])
# FAM, USE -> ATT
m2 = ols(dat["ATT"], dat[["FAM","USE"]])
# FAM, ATT, USE -> BI
m3 = ols(dat["BI"], dat[["FAM","ATT","USE"]])

print("\n=== Approx. path coefficients (OLS on latent scores) ===")
print("FAM ~ USE")
print(m1.summary().tables[1])
print("\nATT ~ FAM + USE")
print(m2.summary().tables[1])
print("\nBI ~ FAM + ATT + USE")
print(m3.summary().tables[1])

# R²
print("\n=== R-squared ===")
print(f"R2(FAM): {m1.rsquared:.3f}")
print(f"R2(ATT): {m2.rsquared:.3f}")
print(f"R2(BI):  {m3.rsquared:.3f}")

# Loadings (correlation between indicator and its latent score)
print("\n=== Loadings (indicator ↔ latent score) ===")
for lv, inds in blocks.items():
    print(f"\n{lv}")
    for ind in inds:
        r = np.corrcoef(df[ind].dropna(), df[lv].dropna())[0,1]
        print(f"  {ind}: {r:.3f}")

# CR & AVE (using correlations as loadings proxy)
def cr_ave(latent, indicators):
    # proxy loadings: corr(ind, latent)
    lam = [np.corrcoef(df[ind].dropna(), df[latent].dropna())[0,1] for ind in indicators]
    lam = np.array(lam)
    theta = 1 - lam**2
    CR = (lam.sum()**2) / ((lam.sum()**2) + theta.sum())
    AVE = (lam**2).mean()
    return CR, AVE

print("\n=== Reliability (CR & AVE; proxy) ===")
for lv, inds in blocks.items():
    CR, AVE = cr_ave(lv, inds)
    print(f"{lv}: CR={CR:.3f}, AVE={AVE:.3f} (k={len(inds)})")

d:\anaconda\D_anaconda3\lib\site-packages\statsmodels\tools\_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm



=== Approx. path coefficients (OLS on latent scores) ===
FAM ~ USE
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7149      0.157     10.901      0.000       1.406       2.024
USE            0.1116      0.051      2.210      0.028       0.012       0.211

ATT ~ FAM + USE
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.6419      0.068     38.882      0.000       2.508       2.775
FAM            0.0674      0.018      3.797      0.000       0.033       0.102
USE           -0.0216      0.020     -1.102      0.271      -0.060       0.017

BI ~ FAM + ATT + USE
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.0943   

d:\anaconda\D_anaconda3\lib\site-packages\statsmodels\tsa\tsatools.py:117: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
